In [ ]:
!pip -q install ultralytics pyyaml pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.2 MB/s eta 0:00:00


In [ ]:
import math
import shutil
import zipfile
from pathlib import Path

import pandas as pd
import torch
import yaml
from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## RDD2022 YOLOv8 baseline for paper comparison

This notebook is set up to be **closer to the YOLOv8-based road-damage papers** you shared:

- **Model:** `yolov8s.pt` baseline
- **Classes:** 4 classes only (`D00`, `D10`, `D20`, `D40`)
- **Image size:** `640`
- **Optimizer:** `SGD`
- **Epochs:** `200`
- **Split assumption:** train/val from the annotated split, and test from `benchmark_test/images` if present

This should give you a fairer baseline to compare against papers that report YOLOv8-style results on RDD2022.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Update these paths if your zip lives somewhere else on Drive.
DRIVE_ZIP = Path('/content/drive/MyDrive/Research/Dataset/RDD2022_ALL.zip')
LOCAL_ZIP = Path('/content/RDD2022_ALL.zip')
LOCAL_ROOT = Path('/content/RDD2022_ALL')

assert DRIVE_ZIP.exists(), f'Zip not found: {DRIVE_ZIP}'

shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)

if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)

with zipfile.ZipFile(LOCAL_ZIP, 'r') as zf:
    zf.extractall('/content')

print('Done extracting to:', LOCAL_ROOT)


Done extracting to: /content/RDD2022_ALL


In [ ]:
DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/Research/runs_rdd2022')
RUN_NAME = 'yolov26m_rdd2022_ALL'


In [ ]:
dataset_root = LOCAL_ROOT
yaml_path = dataset_root / 'dataset.yaml'
run_dir = DRIVE_RUNS_ROOT / RUN_NAME

print('Using dataset root:', dataset_root)
print('Run directory:', run_dir)


Using dataset root: /content/RDD2022_ALL
Run directory: /content/drive/MyDrive/Research/runs_rdd2022/yolov26m_rdd2022_ALL


In [ ]:
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

device = 0 if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4
Using device: 0


In [ ]:
test_rel = None
if (dataset_root / 'test' / 'images').exists():
    test_rel = 'test/images'
elif (dataset_root / 'test' / 'images').exists():
    test_rel = 'test/images'

data_config = {
    'path': str(dataset_root),
    'train': 'train/images',
    'val': 'val/images',
    'nc': 4,
    'names': [
        'longitudinal crack',
        'transverse crack',
        'alligator crack',
        'pothole'
    ],
}

if test_rel is not None:
    data_config['test'] = test_rel

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, sort_keys=False)

print('Saved yaml to:', yaml_path)
print('Test split set to:', data_config.get('test', 'None'))


Saved yaml to: /content/RDD2022_ALL/dataset.yaml
Test split set to: test/images


In [ ]:
for split in ['train', 'val', 'test']:
    img_dir = dataset_root / split / 'images'
    lbl_dir = dataset_root / split / 'labels'
    print(f'\n[{split}]')
    print('images dir exists:', img_dir.exists())
    print('labels dir exists:', lbl_dir.exists())
    print('image count:', len(list(img_dir.glob('*'))))
    print('label count:', len(list(lbl_dir.glob('*.txt'))))

# if 'test' in data_config:
#     test_img_dir = dataset_root / data_config['test']
#     print('\n[test]')
#     print('images dir exists:', test_img_dir.exists())
#     print('image count:', len(list(test_img_dir.glob('*'))))



[train]
images dir exists: True
labels dir exists: True
image count: 15294
label count: 15294

[val]
images dir exists: True
labels dir exists: True
image count: 4370
label count: 4370

[test]
images dir exists: True
labels dir exists: True
image count: 2184
label count: 2184


In [ ]:
all_classes = set()

for split in ['train', 'val', 'test']:
    lbl_dir = dataset_root / split / 'labels'

    if not lbl_dir.exists():
        continue

    for label_file in lbl_dir.glob('*.txt'):
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 1:
                    cls_id = int(float(parts[0]))
                    all_classes.add(cls_id)

print("\nTotal unique classes:", len(all_classes))
print("Class IDs:", sorted(all_classes))


Total unique classes: 4
Class IDs: [0, 1, 2, 3]


In [ ]:
from pathlib import Path

# dataset_root = Path("/content/drive/MyDrive/your_dataset")  # set yours

for split in ["train_set", "val_set", "test_set"]:
    img_dir = dataset_root / split / "images"
    lbl_dir = dataset_root / split / "labels"

    image_files = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
        image_files.extend(img_dir.glob(ext))

    bg_images = []

    for img_path in image_files:
        label_path = lbl_dir / f"{img_path.stem}.txt"

        # Background image = no label file OR empty label file
        if not label_path.exists() or label_path.stat().st_size == 0:
            bg_images.append(img_path.name)

    print(f"\n[{split}]")
    print("total images:", len(image_files))
    print("background images:", len(bg_images))
    print("object images:", len(image_files) - len(bg_images))

print("\nDone.")


[train_set]
total images: 0
background images: 0
object images: 0

[val_set]
total images: 0
background images: 0
object images: 0

[test_set]
total images: 0
background images: 0
object images: 0

Done.


In [ ]:
model = YOLO('yolo26m')

### Training notes

These settings are chosen to be much closer to the YOLOv8 paper-style baselines than your previous YOLO26 notebook:

- `imgsz=640`
- `epochs=200`
- `optimizer='SGD'`
- `lr0=0.01`, `momentum=0.937`
- default augmentation kept on

If batch size 32 is too large on your Colab session, reduce it to 16.


In [ ]:
results = model.train(
    data=str(yaml_path),

    epochs=100,
    imgsz=640,
    batch=32,        # kept as your choice
    device=device,
    workers=16,      # kept as your choice

    project=str(DRIVE_RUNS_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,

    optimizer="SGD",
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,

    # paper-reported / close reproduction choices
    label_smoothing=0.1,
    auto_augment=None,

    # stricter reproduction
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    translate=0.0,
    scale=0.0,
    fliplr=0.0,
    mosaic=0.0,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    mixup=0.0,
    copy_paste=0.0,

    patience=0,
    save=True,
    save_period=5,
    amp=True,
    cache="disk",
    val=True,
    plots=True,
)

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=32, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/RDD2022_ALL/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=yolov26m_rdd2022_ALL, nbs=64, nm

In [ ]:
best_model_path = DRIVE_RUNS_ROOT / RUN_NAME / 'weights' / 'best.pt'
last_model_path = DRIVE_RUNS_ROOT / RUN_NAME / 'weights' / 'last.pt'

print('Best model path:', best_model_path)
print('Last model path:', last_model_path)


Best model path: /content/drive/MyDrive/Research/runs_rdd2022/yolov26s_rdd2022_ALL/weights/best.pt
Last model path: /content/drive/MyDrive/Research/runs_rdd2022/yolov26s_rdd2022_ALL/weights/last.pt


In [ ]:
best_model = YOLO(str(best_model_path))

val_metrics = best_model.val(
    data=str(yaml_path),
    split='test',
    imgsz=640,
    batch=8,
    device=device,
)

print('\n── Validation Results ──')
for i, name in val_metrics.names.items():
    print(f'  {name:<22} mAP50-95: {val_metrics.box.maps[i]:.3f}')
print(f"\n  {'all':<22} Precision: {val_metrics.box.mp:.3f}")
print(f"  {'all':<22} Recall:    {val_metrics.box.mr:.3f}")
print(f"  {'all':<22} mAP50:     {val_metrics.box.map50:.3f}")
print(f"  {'all':<22} mAP50-95:  {val_metrics.box.map:.3f}")

if 'test' in data_config:
    print('\nNote: official benchmark_test images are unlabeled, so standard mAP evaluation is only possible if your test split has labels.')


Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO26s summary (fused): 122 layers, 9,466,728 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 20.2±9.5 MB/s, size: 52.6 KB)
val: Scanning /content/RDD2022_ALL/test/labels... 2184 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2184/2184 832.6it/s 2.6s
val: New cache created: /content/RDD2022_ALL/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 273/273 14.6it/s 18.6s
                   all       2184       5172      0.555      0.428      0.425      0.191
    longitudinal crack       1253       2522      0.583      0.429      0.431      0.212
      transverse crack        669       1030       0.55      0.397        0.4      0.171
       alligator crack        818       1024      0.598      0.542      0.554      0.264
               pothole        356        596       0.49      0.342    

In [ ]:
paper_rows = [
    {'method': 'Your YOLOv8s baseline', 'split': 'current val', 'classes': 4, 'mAP50': float(val_metrics.box.map50), 'mAP50_95': float(val_metrics.box.map)},
    {'method': 'RDD-YOLO (paper)', 'split': 'val', 'classes': 4, 'mAP50': 0.625, 'mAP50_95': 0.364},
    {'method': 'YOLO-RD baseline YOLOv8s', 'split': 'RM-RDD test', 'classes': 8, 'mAP50': 0.648, 'mAP50_95': 0.395},
    {'method': 'YOLO-RD final', 'split': 'RM-RDD test', 'classes': 8, 'mAP50': 0.667, 'mAP50_95': 0.412},
]
compare_df = pd.DataFrame(paper_rows)
compare_df


,method,split,classes,mAP50,mAP50_95
0,Your YOLOv8s baseline,current val,4,0.601035,0.312088
1,RDD-YOLO (paper),val,4,0.625000,0.364000
2,YOLO-RD baseline YOLOv8s,RM-RDD test,8,0.648000,0.395000
3,YOLO-RD final,RM-RDD test,8,0.667000,0.412000


In [ ]:
pred_results = best_model.predict(
    source=str(dataset_root / 'test' / 'images'),
    imgsz=640,
    conf=0.25,
    device=device,
    save=True,
    project=str(DRIVE_RUNS_ROOT),
    name=f'{RUN_NAME}_preds_val',
    exist_ok=True,
)



WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/2184 /content/RDD2022_ALL/test/images/China_MotorBike__China_MotorBike_000019.jpg: 640x640 1 longitudinal crack, 1 transverse crack, 11.7ms
image 2/2184 /content/RDD2022_ALL/test/images/China_MotorBike__China_MotorBike_000038.jpg: 640x640 2 longitudinal cracks, 14.8ms
image 3/2184 /content/RDD2022_ALL/test/images/China_MotorBike__China_MotorBike_000042.jpg: 640x640 4 longitudinal cracks, 1 transverse crack, 12.6ms
image 4/2184 /content/RDD2022_AL

In [ ]:
from google.colab import runtime
runtime.unassign()